# Mohan Sir — Bunny Motion Source Audit

**Metadata-only, read-only audit.** This notebook checks the 39 Bunny Stream videos already catalogued in the Swar Yoga RAG source table. It requests each video's Bunny metadata (title, duration, encoding status, and available resolutions) and saves a CSV report in this Colab session.

It does **not** download/watch video, extract frames or movement, install packages, run AI inference, modify Bunny, upload files, or establish consent/licensing. Bunny's catalog approval label is not a rights determination. Metadata cannot tell whether the person is continuously visible, whether hands are occluded, or whether the footage is useful for motion retargeting. The next step after this report is owner-led visual review of a small authorized sample.

The source catalog baseline was read from Bunny SQL on November 10, 2026: 39 rows, 39 unique Bunny video IDs, all in library 606386; catalog language tags: English 8, Hindi 13, Marathi 18. These counts describe the catalog only, not processed videos or an avatar.

## 1. Review the audit scope

This notebook contains only source IDs already present in the RAG Bunny Stream catalog. No database credentials are needed. To call Bunny's read-only video metadata endpoint, add `BUNNY_STREAM_API_KEY` in Colab's **Secrets** panel (key icon) and allow this notebook to access it. Never paste the key into a cell or share outputs that contain it.

If you do not have permission to inspect these recordings, stop here. The API key is used only in an HTTPS `GET` request to Bunny Stream; it is never printed or saved in the report. The API is queried once per catalogued video with a short delay.

In [ ]:
SOURCE_VIDEOS = [
    {"title": "Marathi-Oct-Eve-Swar Yoga L-1 -2026 — Speaker View — 2026-10-28", "language": "mr", "library_id": "606386", "video_id": "baedb88d-2353-4215-ab33-2ddbbf5cb9d9"},
    {"title": "Marathi-Oct-Eve-Swar Yoga L-1 -2026 — Speaker View — 2026-10-27", "language": "mr", "library_id": "606386", "video_id": "f7efb8bc-7662-4c06-afbc-bda57e09f9e1"},
    {"title": "Marathi-Oct-Eve-Swar Yoga L-1 -2026 — Speaker View — 2026-10-26", "language": "mr", "library_id": "606386", "video_id": "3246a191-7f93-4002-ab2d-4d7916410577"},
    {"title": "Hindi Oct-26 Mor Swar Yoga L-1 — Speaker View — 2026-10-26", "language": "hi", "library_id": "606386", "video_id": "107fce87-d999-4671-9d38-58ee2d3522b2"},
    {"title": "Eng Morning Swar Yoga — Speaker View — 2026-10-26", "language": "en", "library_id": "606386", "video_id": "4cd25a86-3ee2-41ce-bb41-e38cf946d6aa"},
    {"title": "Hindi Oct-26 Mor Swar Yoga L-1 — Speaker View — 2026-10-22", "language": "hi", "library_id": "606386", "video_id": "9ced540f-c85e-4acc-a744-5ff72cdcb96d"},
    {"title": "Eng Morning Swar Yoga — Speaker View — 2026-10-22", "language": "en", "library_id": "606386", "video_id": "6902ac81-474b-4083-a64b-1d403e77244d"},
    {"title": "Marathi-Oct-Eve-Swar Yoga L-1 -2026 — Speaker View — 2026-10-21", "language": "mr", "library_id": "606386", "video_id": "4e3dfe2f-1ab2-4ece-ab1e-da5ee179c30e"},
    {"title": "Swar-yoga-hindi-SnapYT.App (2).mp4", "language": "hi", "library_id": "606386", "video_id": "f41134e9-ac7a-4495-98d9-27cc7a3f4784"},
    {"title": "Swar-yoga-hindi-SnapYT.App (1).mp4", "language": "hi", "library_id": "606386", "video_id": "0e55351e-e368-4623-9d03-f91b620ebe61"},
    {"title": "Swar-Yoga-Hindi-SnapYT.App.mp4", "language": "hi", "library_id": "606386", "video_id": "e2a50b54-e5e2-42bd-b4d3-7b1d6bf8eb42"},
    {"title": "Swar yoga intro clip-3 - 4K.mov", "language": "mr", "library_id": "606386", "video_id": "04f6e33e-1ee3-447d-b1be-70462907ab42"},
    {"title": "Swar yoga intro part-2 - 4K.mov", "language": "mr", "library_id": "606386", "video_id": "ba06f3a9-e19e-4dd6-8ef7-e5effb9b5776"},
    {"title": "Swar yoga intro1 - 4K.mov", "language": "en", "library_id": "606386", "video_id": "247e9982-7723-49bb-a327-d923c67e543a"},
    {"title": "GMT20260917-133158_Recording_gvo_1280x720.mp4", "language": "mr", "library_id": "606386", "video_id": "d6ec2903-1adc-4751-ab12-963b5b0ba08c"},
    {"title": "GMT20260917-133158_Recording_avo_640x360.mp4", "language": "mr", "library_id": "606386", "video_id": "7d0e63ef-1ef7-47ec-9249-92731d10ec75"},
    {"title": "Marker Feature Test (hi)", "language": "en", "library_id": "606386", "video_id": "fc1ce7fd-910c-480c-82b7-4b46b780b66c"},
    {"title": "Meditation workshop — 2026-06-17", "language": "mr", "library_id": "606386", "video_id": "33b599f9-fbfc-432e-9339-b5f7b4a0e5b3"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "250f6db9-9b11-43d1-8eef-c219e892002b"},
    {"title": "Swar Yoga- Evening Batch(I) — 2026-06-09", "language": "mr", "library_id": "606386", "video_id": "9dcae30a-212c-4a3d-bdb5-237b4f66faba"},
    {"title": "Swar Yoga- Evening Batch(I) — 2026-06-10", "language": "mr", "library_id": "606386", "video_id": "8a676611-9175-47e0-bc0e-5e9d12dfbd23"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "b17ecd51-734c-4edc-85bf-8e6eb6aaaace"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "9789b24f-9fcb-470c-aad4-3db3ea404ce6"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "62803d4f-09ba-451a-93ae-4c655594e895"},
    {"title": "Day-2 Water therapy, Hunger, Gasses", "language": "mr", "library_id": "606386", "video_id": "f9fc9a30-bcbe-4070-a3d8-6dd40ef9f02a"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "ec4e44cd-341f-4070-bd70-31c2b278d417"},
    {"title": "Day 1 all itrodotion", "language": "en", "library_id": "606386", "video_id": "c16ba961-5625-47bf-9193-dbec875673a7"},
    {"title": "Swar yoga Day-1", "language": "en", "library_id": "606386", "video_id": "5fd3882c-d768-4335-a33e-229a057f37d7"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "a269f15c-6871-4eeb-8420-82f62c7a5b60"},
    {"title": "7 days Swar Yoga Hindi", "language": "hi", "library_id": "606386", "video_id": "74f27512-8247-44c1-9558-4c2d54c84e5f"},
    {"title": "Swar Aahar Shastra", "language": "mr", "library_id": "606386", "video_id": "a09f45d0-ccf2-4a1a-b581-6a33a455fa4b"},
    {"title": "Swar Aahar Shastra", "language": "mr", "library_id": "606386", "video_id": "f1c34f2e-8415-46ee-94a3-efe82f4ad52f"},
    {"title": "Swar Aahar Shastra", "language": "mr", "library_id": "606386", "video_id": "137b14ff-e134-4d12-b85b-1a9dfd0449cf"},
    {"title": "Swar Aahar Shastra", "language": "mr", "library_id": "606386", "video_id": "2368fa4f-134a-43c1-870a-c5750c65f5da"},
    {"title": "sadhana - april 26.mov", "language": "mr", "library_id": "606386", "video_id": "c828f318-88d6-48e4-b0c9-fe5004ae59f0"},
    {"title": "Swar Aahar Shastra", "language": "en", "library_id": "606386", "video_id": "8123359c-433d-43e2-8f0f-48347e606eac"},
    {"title": "Swar Aahar Shastra", "language": "mr", "library_id": "606386", "video_id": "22f3e9d3-f0d7-4c3b-b344-8d7ed1d0d09c"},
    {"title": "Global Community", "language": "en", "library_id": "606386", "video_id": "d06767ad-3ba3-4695-b3e5-32e3012fdbc2"},
    {"title": "Day-1 all introduction,.mp4", "language": "hi", "library_id": "606386", "video_id": "bb595ddc-1e86-45b0-928e-a75106e608ae"}
]

assert len(SOURCE_VIDEOS) == 39, f"Expected 39 catalogued sources, found {len(SOURCE_VIDEOS)}"
assert len({item['video_id'] for item in SOURCE_VIDEOS}) == len(SOURCE_VIDEOS), "Duplicate video IDs in manifest"
print("Source manifest validated:", len(SOURCE_VIDEOS), "unique catalogued video IDs")
print("Language tags:", {code: sum(item['language'] == code for item in SOURCE_VIDEOS) for code in ('en', 'hi', 'mr')})

## 2. Run the read-only metadata lookup

Run this cell only if you have authorized access to the Bunny library. It makes one bounded `GET` request per source ID and writes `mohan_bunny_motion_source_metadata.csv` to this Colab session. This is metadata inspection only—not video viewing, motion analysis, or permission verification.

In [ ]:
import csv
import json
import time
import urllib.error
import urllib.request
from pathlib import Path

try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab and configure its Secrets panel first.") from exc

api_key = userdata.get("BUNNY_STREAM_API_KEY")
if not api_key:
    raise RuntimeError("BUNNY_STREAM_API_KEY is missing from Colab Secrets. Add it there; do not paste it in this cell.")

def get_video_metadata(source):
    path = f"/library/{source['library_id']}/videos/{source['video_id']}"
    request = urllib.request.Request(
        "https://video.bunnycdn.com" + path,
        headers={"AccessKey": api_key, "Accept": "application/json"},
        method="GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            if response.status != 200:
                raise RuntimeError(f"Bunny returned HTTP {response.status}")
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        if exc.code in (401, 403):
            raise RuntimeError(f"Bunny denied metadata access (HTTP {exc.code}); verify the key has read access to library {source['library_id']}.") from None
        raise RuntimeError(f"Bunny metadata request failed (HTTP {exc.code}) for {source['video_id']}.") from None
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Network error while reading Bunny metadata for {source['video_id']}: {exc.reason}") from None

    resolutions = data.get("availableResolutions", "")
    if isinstance(resolutions, str):
        resolutions = ", ".join(part.strip() for part in resolutions.split(",") if part.strip())
    else:
        resolutions = ""
    return {
        "title": source["title"],
        "catalog_language": source["language"],
        "video_id": source["video_id"],
        "duration_seconds": data.get("length"),
        "width": data.get("width"),
        "height": data.get("height"),
        "available_resolutions": resolutions,
        "encoding_status": data.get("status"),
        "date_uploaded": data.get("dateUploaded"),
        "metadata_result": "ok",
    }

rows = []
for index, source in enumerate(SOURCE_VIDEOS, start=1):
    row = get_video_metadata(source)
    rows.append(row)
    print(f"[{index}/{len(SOURCE_VIDEOS)}] metadata received — {source['title']}")
    time.sleep(0.25)

report_path = Path("mohan_bunny_motion_source_metadata.csv")
fieldnames = list(rows[0].keys()) if rows else []
with report_path.open("w", newline="", encoding="utf-8-sig") as report_file:
    writer = csv.DictWriter(report_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"\nMetadata-only report saved: {report_path} ({len(rows)} rows)")
print("The Bunny API key was not printed or written to the report.")

## 3. What metadata can and cannot tell us

The CSV can help shortlist videos by duration and available resolution. It does **not** report dependable motion-capture suitability: a title containing “Speaker View” does not prove that the full torso, arms, or hands stay visible, and resolution does not prove visual quality or smooth motion.

Next, manually preview one owner-approved candidate in Bunny and check: one clearly visible subject; front/near-front camera; head, shoulders, elbows, wrists and hands in frame; few occlusions/cuts; stable lighting/focus; and usable motion throughout the selected time. Record the chosen video ID and time range. Do not download or extract frames until that specific sample is approved for the next processing step.

No motion has been extracted by this notebook. Do not describe the current body mesh as animated, the source catalog as reviewed, or any result as HD/realistic. The separate multilingual lip-sync notebook is preflight-only and does not render.